# 06 - Hybrid Model

Three models built so far, each with a clearly different job based on what Phase 5 actually
showed:

- **Personalized Frequency** — 0.2838 precision@10, by far the strongest signal, because it
  directly captures the reorder pattern that dominates most baskets
- **ALS** — 0.0894 at its best tuning, weaker alone but structurally different from frequency
- **Content-Based** — 0.0028, deliberately scored on new-item discovery only, and confirmed
  to overlap with ALS at just 0.0246 Jaccard — the two are finding largely different products

Given that gap, a proportional blend of all three would drown ALS and content-based's
contribution down to nothing, while a flat union of top-N lists would let two weak models
dilute the one strong one. The hybrid here instead uses Personalized Frequency as the base
ranking and treats ALS/content-based as **boosts**: if either of them also recommends a
product already in the base list, that product's score goes up and it can move higher in the
final ranking. Products the base model never surfaced at all get appended after it, since
there's no frequency signal to rank them against.

Logic in `src/hybrid_model.py`.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from data_processing import load_raw_data
from eda import build_transactions
from baseline_models import PersonalizedFrequencyModel, get_eval_users, evaluate_model
from collaborative_filtering import ALSModel
from content_based import ContentBasedModel
from hybrid_model import HybridModel

## Load data

In [2]:
processed_path = Path.cwd().parent / "data" / "processed" / "transactions.parquet"

if processed_path.exists():
    txn = pd.read_parquet(processed_path)
else:
    data = load_raw_data()
    txn = build_transactions(data)

data = load_raw_data()  # content-based needs the raw tables directly
eval_users = get_eval_users(txn)
print(txn.shape, len(eval_users), "eval users")

(33819106, 15) 131209 eval users


## Fit all three sub-models

ALS uses `alpha=15`, the value that won the sweep in Phase 5.

In [3]:
pf_model = PersonalizedFrequencyModel(top_n=50).fit(txn)
als_model = ALSModel(factors=50, regularization=0.01, alpha=15.0, iterations=15).fit(txn)
cb_model = ContentBasedModel(top_n=50).fit(txn, data)

C:\Users\shubh\AppData\Roaming\Python\Python314\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

## Combine into the hybrid

Boost weights start at 0.15 each — enough to move a product several ranks when both signals
agree with the base list, without letting a single boosted item leapfrog the top of a strong
Personalized Frequency ranking outright.

In [4]:
hybrid_model = HybridModel(top_n=50, als_boost=0.15, cb_boost=0.15).fit(
    pf_model, als_model, cb_model
)

sample_user = eval_users.iloc[0]["user_id"]
print("PF alone:", pf_model.recommend(sample_user)[:10])
print("Hybrid:  ", hybrid_model.recommend(sample_user)[:10])

PF alone: [196, 12427, 10258, 25133, 13032, 46149, 49235, 13176, 26405, 26088]
Hybrid:   [196, 12427, 10258, 46149, 25133, 49235, 13032, 13176, 26405, 38928]


## Evaluate

Same Precision@10 / Recall@10 harness as every model before it. The bar to clear is
Personalized Frequency's 0.2838 / 0.3298 — if the hybrid can't at least match that while
adding coverage from ALS/content-based, the boosts aren't earning their place.

In [5]:
K = 10
hybrid_results = evaluate_model(hybrid_model.recommend, eval_users, k=K)
hybrid_results

{'k': 10,
 'n_users_evaluated': 131209,
 'precision_at_k': 0.27612282694022516,
 'recall_at_k': 0.3302631496990789}

In [6]:
pd.DataFrame([hybrid_results], index=["Hybrid"]).to_csv(
    Path.cwd().parent / "data" / "processed" / "hybrid_results.csv"
)

## All models side by side

Pulling every result file written across Baseline, Collaborative / ALS, Content And Hybrid into one table — this is the summary
that actually belongs in the project writeup.

In [7]:
processed_dir = Path.cwd().parent / "data" / "processed"

all_results = pd.concat([
    pd.read_csv(processed_dir / "baseline_results.csv", index_col=0),
    pd.read_csv(processed_dir / "als_results.csv", index_col=0),
    pd.read_csv(processed_dir / "content_based_results.csv", index_col=0),
    pd.read_csv(processed_dir / "hybrid_results.csv", index_col=0),
])

all_results[["precision_at_k", "recall_at_k"]].sort_values("precision_at_k", ascending=False)

,precision_at_k,recall_at_k
Personalized Frequency,0.283836,0.329784
Hybrid,0.276123,0.330263
Popularity,0.072522,0.069843
Reorder Popularity,0.072151,0.069616
ALS,0.065520,0.098105
Content-Based,0.002762,0.002989


### Findings

The hybrid model combines the three signals established in the previous modeling phases: **Personalized Frequency** as the primary reorder signal, **ALS** as a collaborative signal, and **Content-Based** recommendations as a product-discovery signal. Because Personalized Frequency is substantially stronger than the other two models, the hybrid uses it as the base ranking and applies ALS and content-based recommendations as targeted score boosts rather than blending the three models equally.

* **Personalized Frequency remains the strongest precision-oriented signal.** It achieves **Precision@10 = 0.2838** and **Recall@10 = 0.3298**, confirming the earlier finding that historical purchase behavior is the dominant signal for this dataset.

* **The hybrid model achieves slightly higher recall.** The hybrid reaches **Precision@10 = 0.2761** and **Recall@10 = 0.3303** across **131,209 evaluation users**. Recall therefore improves marginally over Personalized Frequency, while precision decreases by approximately 0.0077. This indicates that the additional ALS and content-based signals introduce some relevant products that the frequency model alone does not retrieve.

* **The hybrid preserves most of the baseline performance.** Although precision is slightly lower than Personalized Frequency, the hybrid remains substantially stronger than Popularity, Reorder Popularity, ALS, and Content-Based models across the evaluated metrics. This suggests that using Personalized Frequency as the base ranking successfully prevents the weaker complementary models from dominating the recommendations.

* **ALS and Content-Based models provide complementary rather than primary signals.** The previous phases showed that ALS and Content-Based individually perform much worse than Personalized Frequency, while their recommendation lists have limited overlap. The hybrid therefore uses these models to influence products already supported by the base ranking rather than treating them as independent competing recommenders.

* **The hybrid improves coverage of relevant items at a small precision cost.** The difference between Personalized Frequency and Hybrid indicates a trade-off: the hybrid retrieves a slightly greater proportion of the relevant basket while introducing some additional recommendations that are less likely to be correct individually.

* **Modeling implication:** Personalized Frequency should remain the **core recommendation signal**, while ALS and Content-Based models are best retained as complementary signals for discovery and cross-sell opportunities. The current results justify the hybrid architecture, but further tuning of the boost weights should be evaluated to determine whether the recall improvement can be retained while reducing the precision loss.
